# Olist: a logística como freio da satisfação e da recompra

**Tech Challenge — Fase 1 · FIAP Pós Tech Data Analytics**

**Objetivo:** construir um relatório executivo para investidores e acionistas a partir do dataset público de e-commerce da Olist (~100 mil pedidos, 2016–2018), com recomendações acionáveis e previsão.

**Tese:** *a Olist cresceu rápido, mas os atrasos logísticos derrubam a satisfação e a recompra. Corrigir a entrega nas rotas críticas destrava valor.*

**Como reproduzir:** abra no Google Colab e use *Ambiente de execução → Executar tudo*. Os dados são baixados automaticamente do Kaggle (`olistbr/brazilian-ecommerce`) e os dados externos, da API do IBGE. Ao final, todas as tabelas para o Power BI, as figuras e a documentação ficam nas pastas `data/processed/`, `reports/figures/` e `docs/`, e um `.zip` com tudo é gerado.

| Seção | Conteúdo |
|---|---|
| 1 | Configuração e carregamento |
| 2 | Visão geral das tabelas |
| 3 | Diagnóstico de qualidade |
| 4 | Tratamento |
| 5 | Base analítica (1 linha por pedido) |
| 6 | Métricas derivadas |
| 7 | Registro de qualidade e decisões |
| 8 | Crescimento e receita |
| 9 | Logística |
| 10 | Satisfação |
| 11 | Clientes: recompra, coortes e RFM |
| 12 | Previsão |
| 13 | Exportação para o Power BI |
| 14 | Principais números para o relatório |

## 1. Configuração e carregamento

In [ ]:
import os, re, sys, json, shutil, subprocess, unicodedata, urllib.request, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import kagglehub

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Pastas de saída (estrutura do repositório)
PASTA_PROCESSED = "data/processed"
PASTA_FIGURAS = "reports/figures"
PASTA_DOCS = "docs"
for p in (PASTA_PROCESSED, PASTA_FIGURAS, PASTA_DOCS):
    os.makedirs(p, exist_ok=True)

# Período de análise (justificado na seção 3)
DATA_INICIO, DATA_FIM = "2017-01-01", "2018-08-31"

# Identidade visual: azul = padrão, laranja = destaque/problema, cinza = contexto
AZUL, LARANJA, VERDE_AGUA, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#b4b2a9"
plt.rcParams.update({
    "figure.figsize": (10, 5), "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
})

MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]

def num(x, casas=1):
    """Número no padrão brasileiro: 1.234,5"""
    return f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")

def pct(x, casas=0):
    return num(x * 100, casas) + "%"

def rotulo_mes(ts):
    return f"{MESES[ts.month - 1]}/{str(ts.year)[2:]}"

def salvar(fig, nome):
    fig.savefig(os.path.join(PASTA_FIGURAS, f"{nome}.png"))
    plt.show()

# Registro de qualidade: cada problema encontrado e a decisão tomada (vira a seção 7 e o docs/qualidade_dados.md)
registro = []
def registrar(tabela, problema, linhas_afetadas, decisao, justificativa):
    registro.append({"tabela": tabela, "problema": problema, "linhas_afetadas": int(linhas_afetadas),
                     "decisao": decisao, "justificativa": justificativa})

# Números-chave que alimentam o relatório e os slides (seção 14)
achados = {}

In [ ]:
# Download do dataset oficial (público). Se o Colab pedir credenciais, rode kagglehub.login() antes.
caminho = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Pasta dos dados:", caminho)
print(sorted(os.listdir(caminho)))

In [ ]:
arquivos = {
    "orders":    "olist_orders_dataset.csv",
    "items":     "olist_order_items_dataset.csv",
    "payments":  "olist_order_payments_dataset.csv",
    "reviews":   "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers":   "olist_sellers_dataset.csv",
    "products":  "olist_products_dataset.csv",
    "geo":       "olist_geolocation_dataset.csv",
    "cat_tr":    "product_category_name_translation.csv",
}
dfs = {nome: pd.read_csv(os.path.join(caminho, arq)) for nome, arq in arquivos.items()}

orders, items, payments, reviews = dfs["orders"], dfs["items"], dfs["payments"], dfs["reviews"]
customers, sellers, products = dfs["customers"], dfs["sellers"], dfs["products"]
geo, cat_tr = dfs["geo"], dfs["cat_tr"]

for nome, df in dfs.items():
    print(f"{nome:<10} {df.shape[0]:>9,} linhas  {df.shape[1]:>2} colunas")

## 2. Visão geral das tabelas

O "RG" de cada tabela: tamanho, nulos e linhas duplicadas. Esta tabela é o ponto de partida do dicionário de dados.

In [ ]:
def resumo_tabela(nome, df):
    nulos = df.isnull().sum()
    return {"tabela": nome, "linhas": len(df), "colunas": df.shape[1],
            "colunas_com_nulos": int((nulos > 0).sum()), "total_nulos": int(nulos.sum()),
            "linhas_duplicadas": int(df.duplicated().sum())}

resumo = pd.DataFrame([resumo_tabela(n, d) for n, d in dfs.items()])
resumo

In [ ]:
# Onde estão os nulos (só colunas com algum nulo)
nulos = pd.concat({n: d.isnull().sum() for n, d in dfs.items()}).rename("nulos").reset_index()
nulos.columns = ["tabela", "coluna", "nulos"]
nulos = nulos[nulos["nulos"] > 0]
nulos["pct"] = nulos.apply(lambda r: r["nulos"] / len(dfs[r["tabela"]]), axis=1)
nulos.sort_values("nulos", ascending=False)

## 3. Diagnóstico de qualidade

Aqui só **observamos e anotamos**. Cada achado vira uma decisão registrada e aplicada na seção 4.

Primeiro, as datas são convertidas de texto para datetime, o que é necessário para diagnosticar o período e a coerência dos prazos.

In [ ]:
COLS_DATA = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in COLS_DATA:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")
registrar("orders/reviews", "Datas armazenadas como texto", len(orders), "Convertidas para datetime",
          "Necessário para calcular prazos, atrasos e séries mensais")
orders[COLS_DATA].dtypes

In [ ]:
# 3.1 Chaves e granularidade
dup_orders = orders["order_id"].duplicated().sum()
rev_por_pedido = reviews["order_id"].value_counts()
itens_por_pedido = items["order_id"].value_counts()
pag_por_pedido = payments["order_id"].value_counts()

print(f"order_id duplicado em orders:          {dup_orders}")
print(f"Pedidos com mais de 1 avaliação:       {(rev_por_pedido > 1).sum():,}")
print(f"Pedidos com mais de 1 item:            {(itens_por_pedido > 1).sum():,} ({pct((itens_por_pedido > 1).mean(), 1)})")
print(f"Pedidos com mais de 1 pagamento:       {(pag_por_pedido > 1).sum():,} ({pct((pag_por_pedido > 1).mean(), 1)})")

registrar("reviews", "Pedidos com mais de uma avaliação", (rev_por_pedido > 1).sum(),
          "Mantida a avaliação mais recente por pedido", "Garante 1 nota por pedido e evita peso duplo")
registrar("items", "Vários itens por pedido (1 linha por item)", (itens_por_pedido > 1).sum(),
          "Agregado por pedido antes do join", "Evita multiplicar linhas e inflar a receita")
registrar("payments", "Vários pagamentos por pedido (parcelas, vouchers)", (pag_por_pedido > 1).sum(),
          "Agregado por pedido antes do join", "Evita multiplicar linhas e inflar o valor pago")

In [ ]:
# 3.2 Status dos pedidos
status = orders["order_status"].value_counts()
print((status / status.sum()).map(lambda x: pct(x, 2)))

In [ ]:
# 3.3 Distribuição no tempo: as pontas do período têm poucos pedidos
por_mes = orders.set_index("order_purchase_timestamp").resample("MS")["order_id"].count()
fora = por_mes[(por_mes.index < DATA_INICIO) | (por_mes.index > DATA_FIM)]

fig, ax = plt.subplots(figsize=(11, 4))
cores = [CINZA if (m < pd.Timestamp(DATA_INICIO) or m > pd.Timestamp(DATA_FIM)) else AZUL for m in por_mes.index]
ax.bar(range(len(por_mes)), por_mes.values, color=cores, width=0.8)
ax.set_xticks(range(len(por_mes)), [rotulo_mes(m) for m in por_mes.index], rotation=90)
ax.set_title("Meses fora de jan/17–ago/18 (cinza) têm volume residual e foram excluídos")
ax.set_ylabel("Pedidos")
salvar(fig, "00_periodo_analise")

registrar("orders", "Pontas do período com volume residual (set–dez/2016 e set–out/2018)", fora.sum(),
          f"Recorte de {DATA_INICIO} a {DATA_FIM} pela data da compra",
          "Meses incompletos distorcem tendências e comparações")

In [ ]:
# 3.4 Coerência das datas
d = orders
entregue_sem_data = ((d["order_status"] == "delivered") & d["order_delivered_customer_date"].isna()).sum()
entrega_antes_compra = (d["order_delivered_customer_date"] < d["order_purchase_timestamp"]).sum()
postagem_antes_aprov = (d["order_delivered_carrier_date"] < d["order_approved_at"]).sum()
entrega_antes_postagem = (d["order_delivered_customer_date"] < d["order_delivered_carrier_date"]).sum()
print(f"'delivered' sem data de entrega:       {entregue_sem_data}")
print(f"Entrega antes da compra:               {entrega_antes_compra}")
print(f"Postagem antes da aprovação:           {postagem_antes_aprov}")
print(f"Entrega antes da postagem:             {entrega_antes_postagem}")

registrar("orders", "Pedidos 'delivered' sem data de entrega", entregue_sem_data,
          "Removidos da base analítica", "Sem data de entrega não é possível medir prazo nem atraso")
registrar("orders", "Etapas com datas fora de ordem (ex.: postagem antes da aprovação)",
          postagem_antes_aprov + entrega_antes_postagem,
          "Pedido mantido; tempo da etapa incoerente tratado como vazio",
          "Erro de registro na etapa não invalida o prazo total do pedido")

In [ ]:
# 3.5 Clientes, produtos, pagamentos e geolocalização
n_cid, n_cuid = customers["customer_id"].nunique(), customers["customer_unique_id"].nunique()
print(f"customer_id distintos:        {n_cid:,}")
print(f"customer_unique_id distintos: {n_cuid:,}  → {n_cid - n_cuid:,} IDs repetem a mesma pessoa")

sem_cat = products["product_category_name"].isna().sum()
sem_trad = (~products["product_category_name"].dropna().isin(cat_tr["product_category_name"])).sum()
pag_indef = (payments["payment_type"] == "not_defined").sum()
pag_zero = (payments["payment_value"] == 0).sum()
geo_dup = geo["geolocation_zip_code_prefix"].duplicated().sum()
print(f"Produtos sem categoria:       {sem_cat}")
print(f"Produtos com categoria sem tradução: {sem_trad}")
print(f"Pagamentos 'not_defined':     {pag_indef} | pagamentos de valor zero: {pag_zero}")
print(f"Linhas de geolocalização repetindo CEP: {geo_dup:,}")

registrar("customers", "customer_id é gerado a cada pedido, não identifica a pessoa", n_cid - n_cuid,
          "customer_unique_id usado para recompra, coortes e RFM", "Sem isso a recompra apareceria como zero")
registrar("products", "Produtos sem categoria", sem_cat, "Categoria preenchida como 'sem categoria'",
          "Mantém os pedidos na análise sem inventar categoria")
registrar("products", "Colunas com erro de digitação ('lenght')", 2, "Renomeadas para 'length'", "Padronização")
registrar("payments", "Pagamentos 'not_defined' ou de valor zero", pag_indef + pag_zero,
          "Mantidos; receita calculada pelo preço dos itens", "Receita não depende da tabela de pagamentos")
registrar("geo", "Vários registros de coordenadas por CEP", geo_dup,
          "Tabela não utilizada; análise regional feita por UF", "UF é suficiente e evita duplicação no join")

## 4. Tratamento

Uma decisão por célula. Todas estão registradas na seção 7.

In [ ]:
# 4.1 Recorte de período
pedidos_periodo = orders[(orders["order_purchase_timestamp"] >= DATA_INICIO) &
                         (orders["order_purchase_timestamp"] < pd.Timestamp(DATA_FIM) + pd.Timedelta(days=1))].copy()
print(f"Pedidos no período: {len(pedidos_periodo):,} de {len(orders):,}")

# Guardamos os status do período: cancelamentos e indisponibilidades são informação para o relatório
status_periodo = pedidos_periodo["order_status"].value_counts(normalize=True)
achados["taxa_cancelamento"] = float(status_periodo.get("canceled", 0) + status_periodo.get("unavailable", 0))
print(f"Cancelados + indisponíveis no período: {pct(achados['taxa_cancelamento'], 2)}")

In [ ]:
# 4.2 Apenas pedidos entregues e com data de entrega
nao_entregues = (pedidos_periodo["order_status"] != "delivered").sum()
pedidos = pedidos_periodo[pedidos_periodo["order_status"] == "delivered"].copy()
pedidos = pedidos.dropna(subset=["order_delivered_customer_date"])
pedidos = pedidos[pedidos["order_delivered_customer_date"] >= pedidos["order_purchase_timestamp"]]
registrar("orders", "Pedidos não entregues (cancelados, em trânsito etc.)", nao_entregues,
          "Fora da base analítica; taxa reportada à parte", "Só pedidos entregues permitem medir prazo e satisfação")
print(f"Pedidos entregues no período: {len(pedidos):,}")

In [ ]:
# 4.3 Produtos: nomes de colunas, categoria e tradução
products = products.rename(columns={"product_name_lenght": "product_name_length",
                                    "product_description_lenght": "product_description_length"})
products["product_category_name"] = products["product_category_name"].fillna("sem_categoria")
products = products.merge(cat_tr, on="product_category_name", how="left")
products["product_category_name_english"] = products["product_category_name_english"].fillna(products["product_category_name"])
products["categoria"] = products["product_category_name"].str.replace("_", " ").str.capitalize()

In [ ]:
# 4.4 Avaliações: 1 por pedido (a mais recente)
reviews_1 = (reviews.sort_values("review_answer_timestamp")
                    .drop_duplicates("order_id", keep="last"))
print(f"Avaliações: {len(reviews):,} → {len(reviews_1):,} (1 por pedido)")

## 5. Base analítica (1 linha por pedido)

Regra de ouro: **resumir antes de juntar**. `items` e `payments` têm várias linhas por pedido; são agregados primeiro.

**Definição de receita:** soma do preço dos itens (GMV de produtos), sem frete. O frete é analisado à parte.

In [ ]:
# Itens → por pedido. Seller e categoria "principais" = os do item mais caro.
itens_ord = (items.merge(products[["product_id", "categoria", "product_category_name_english"]], on="product_id", how="left")
                  .sort_values(["order_id", "price"], ascending=[True, False]))
agg_itens = itens_ord.groupby("order_id").agg(
    valor_produtos=("price", "sum"), valor_frete=("freight_value", "sum"),
    qtd_itens=("order_item_id", "count"), qtd_sellers=("seller_id", "nunique"),
    seller_id=("seller_id", "first"), categoria=("categoria", "first"),
    categoria_en=("product_category_name_english", "first"),
).reset_index()

# Pagamentos → por pedido. Forma de pagamento "principal" = a de maior valor.
agg_pag = (payments.sort_values(["order_id", "payment_value"], ascending=[True, False])
                   .groupby("order_id").agg(valor_pago=("payment_value", "sum"),
                                            forma_pagamento=("payment_type", "first"),
                                            parcelas=("payment_installments", "max"),
                                            qtd_pagamentos=("payment_sequential", "count")).reset_index())

base = (pedidos
        .merge(customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]], on="customer_id", how="left")
        .merge(agg_itens, on="order_id", how="left")
        .merge(agg_pag, on="order_id", how="left")
        .merge(reviews_1[["order_id", "review_score", "review_comment_message"]], on="order_id", how="left")
        .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left"))

assert len(base) == len(pedidos), "Algum merge duplicou linhas!"
assert base["order_id"].is_unique

sem_itens = base["valor_produtos"].isna().sum()
base = base.dropna(subset=["valor_produtos"])
registrar("items", "Pedidos entregues sem itens registrados", sem_itens, "Removidos",
          "Sem itens não há receita, categoria nem seller")
print(f"Base analítica: {len(base):,} pedidos × {base.shape[1]} colunas")

## 6. Métricas derivadas

In [ ]:
def dias_entre(inicio, fim):
    return (base[fim] - base[inicio]).dt.total_seconds() / 86400

base["dias_entrega"] = dias_entre("order_purchase_timestamp", "order_delivered_customer_date")
base["dias_aprovacao"] = dias_entre("order_purchase_timestamp", "order_approved_at")
base["dias_postagem"] = dias_entre("order_approved_at", "order_delivered_carrier_date")
base["dias_transporte"] = dias_entre("order_delivered_carrier_date", "order_delivered_customer_date")
for c in ["dias_aprovacao", "dias_postagem", "dias_transporte"]:
    base.loc[base[c] < 0, c] = np.nan  # etapa com datas fora de ordem (ver seção 3.4)

base["dias_prometidos"] = dias_entre("order_purchase_timestamp", "order_estimated_delivery_date")
base["dias_atraso"] = (base["order_delivered_customer_date"].dt.normalize()
                       - base["order_estimated_delivery_date"].dt.normalize()).dt.days
base["atrasado"] = base["dias_atraso"] > 0
base["faixa_atraso"] = pd.cut(base["dias_atraso"], [-np.inf, 0, 3, 7, 14, np.inf],
                              labels=["No prazo", "1–3 dias", "4–7 dias", "8–14 dias", "15+ dias"])

base["frete_pct"] = base["valor_frete"] / base["valor_produtos"]
base["mes"] = base["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()

REGIAO = {**dict.fromkeys(["SP", "RJ", "MG", "ES"], "Sudeste"), **dict.fromkeys(["PR", "SC", "RS"], "Sul"),
          **dict.fromkeys(["DF", "GO", "MT", "MS"], "Centro-Oeste"),
          **dict.fromkeys(["BA", "PE", "CE", "MA", "PB", "PI", "RN", "AL", "SE"], "Nordeste"),
          **dict.fromkeys(["PA", "AM", "TO", "RO", "AC", "AP", "RR"], "Norte")}
base["regiao_cliente"] = base["customer_state"].map(REGIAO)
base["regiao_seller"] = base["seller_state"].map(REGIAO)
base["rota"] = base["seller_state"] + "→" + base["customer_state"]
base["interestadual"] = base["seller_state"] != base["customer_state"]
base["nota_baixa"] = base["review_score"] <= 2

base[["dias_entrega", "dias_prometidos", "dias_atraso", "frete_pct"]].describe().T

In [ ]:
# Checagens finais da base
assert base["order_id"].is_unique
assert (base["dias_entrega"] >= 0).all()
soma_itens = items.loc[items["order_id"].isin(base["order_id"]), "price"].sum()
assert np.isclose(base["valor_produtos"].sum(), soma_itens), "Receita não bate com a tabela de itens"
print("Checagens OK:")
print(f"  {len(base):,} pedidos únicos | receita de R$ {num(base['valor_produtos'].sum() / 1e6, 2)} mi bate com items")
print(f"  {pct(base['review_score'].notna().mean(), 1)} dos pedidos têm avaliação")

## 7. Registro de qualidade e decisões

Peça central de governança: cada problema encontrado, quantas linhas afetou, o que foi feito e por quê. É salvo em `docs/qualidade_dados.md`.

In [ ]:
registro_df = pd.DataFrame(registro)
registro_df

## 8. Crescimento e receita

Contexto para o investidor: quanto a Olist cresceu, onde está a receita e onde há espaço para crescer.

In [ ]:
mensal = base.groupby("mes").agg(pedidos=("order_id", "count"), receita=("valor_produtos", "sum"),
                                 ticket=("valor_produtos", "mean"), pct_atraso=("atrasado", "mean"),
                                 nota=("review_score", "mean")).reset_index()

m = mensal.set_index("mes")
jan_ago_17 = m.loc["2017-01":"2017-08", "pedidos"].sum()
jan_ago_18 = m.loc["2018-01":"2018-08", "pedidos"].sum()
cresc = jan_ago_18 / jan_ago_17 - 1
rec_17, rec_18 = m.loc["2017-01":"2017-08", "receita"].sum(), m.loc["2018-01":"2018-08", "receita"].sum()
achados.update(pedidos_total=int(len(base)), receita_total=float(base["valor_produtos"].sum()),
               ticket_medio=float(base["valor_produtos"].mean()), crescimento_pedidos_yoy=float(cresc),
               crescimento_receita_yoy=float(rec_18 / rec_17 - 1))
mes_pico = m["pedidos"].idxmax()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(range(len(m)), m["pedidos"], color=AZUL, lw=2, marker="o", ms=4)
i_pico = list(m.index).index(mes_pico)
ax.annotate(f"Pico: {rotulo_mes(mes_pico)}\n{num(m.loc[mes_pico, 'pedidos'], 0)} pedidos",
            (i_pico, m.loc[mes_pico, "pedidos"]), xytext=(-110 if i_pico > 4 else 20, -5), textcoords="offset points",
            arrowprops=dict(arrowstyle="-", color="#52514e"), fontsize=9, va="top")
ax.set_xticks(range(len(m)), [rotulo_mes(x) for x in m.index], rotation=45)
ax.set_ylim(0)
ax.set_title(f"Pedidos cresceram {pct(cresc)} de jan–ago/2017 para jan–ago/2018")
ax.set_ylabel("Pedidos entregues por mês")
salvar(fig, "01_pedidos_mensais")
print(f"Receita: +{pct(rec_18 / rec_17 - 1)} no mesmo comparativo | ticket médio R$ {num(base['valor_produtos'].mean(), 2)}")

In [ ]:
uf = base.groupby("customer_state").agg(pedidos=("order_id", "count"), receita=("valor_produtos", "sum")).sort_values("receita", ascending=False)
uf["share"] = uf["receita"] / uf["receita"].sum()
top_uf = uf.head(10)
achados["share_receita_top_uf"] = float(uf["share"].iloc[0]); achados["top_uf"] = uf.index[0]
achados["share_receita_sudeste"] = float(base.loc[base["regiao_cliente"] == "Sudeste", "valor_produtos"].sum() / base["valor_produtos"].sum())

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(top_uf.index[::-1], top_uf["share"][::-1], color=[CINZA] * 9 + [AZUL])
for y, v in enumerate(top_uf["share"][::-1]):
    ax.text(v + 0.004, y, pct(v, 1), va="center", fontsize=9)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0)); ax.grid(axis="y", visible=False)
ax.set_title(f"{uf.index[0]} concentra {pct(uf['share'].iloc[0])} da receita; o Sudeste, {pct(achados['share_receita_sudeste'])}")
ax.set_xlabel("Participação na receita")
salvar(fig, "02_receita_por_uf")

In [ ]:
cat = base.groupby("categoria")["valor_produtos"].sum().sort_values(ascending=False)
share_top10 = cat.head(10).sum() / cat.sum()
achados["share_receita_top10_categorias"] = float(share_top10)
top_cat = (cat.head(10) / cat.sum())

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(top_cat.index[::-1], top_cat.values[::-1], color=AZUL)
for y, v in enumerate(top_cat.values[::-1]):
    ax.text(v + 0.002, y, pct(v, 1), va="center", fontsize=9)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0)); ax.grid(axis="y", visible=False)
ax.set_title(f"10 categorias geram {pct(share_top10)} da receita (de {len(cat)} no total)")
ax.set_xlabel("Participação na receita")
salvar(fig, "03_receita_por_categoria")

### 8.1 Dados externos: IBGE

População estimada (tabela SIDRA 6579) e PIB (tabela SIDRA 5938) por UF em 2018, para medir **pedidos por 100 mil habitantes** e comparar com a renda. Se a API do IBGE estiver fora do ar, o notebook usa valores de referência embutidos e registra isso.

In [ ]:
COD_UF = {11: "RO", 12: "AC", 13: "AM", 14: "RR", 15: "PA", 16: "AP", 17: "TO", 21: "MA", 22: "PI", 23: "CE",
          24: "RN", 25: "PB", 26: "PE", 27: "AL", 28: "SE", 29: "BA", 31: "MG", 32: "ES", 33: "RJ", 35: "SP",
          41: "PR", 42: "SC", 43: "RS", 50: "MS", 51: "MT", 52: "GO", 53: "DF"}
NOME_UF = {"AC": "Acre", "AL": "Alagoas", "AP": "Amapá", "AM": "Amazonas", "BA": "Bahia", "CE": "Ceará",
           "DF": "Distrito Federal", "ES": "Espírito Santo", "GO": "Goiás", "MA": "Maranhão", "MT": "Mato Grosso",
           "MS": "Mato Grosso do Sul", "MG": "Minas Gerais", "PA": "Pará", "PB": "Paraíba", "PR": "Paraná",
           "PE": "Pernambuco", "PI": "Piauí", "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte",
           "RS": "Rio Grande do Sul", "RO": "Rondônia", "RR": "Roraima", "SC": "Santa Catarina",
           "SP": "São Paulo", "SE": "Sergipe", "TO": "Tocantins"}

# Valores de referência (aproximados) caso a API não responda: população 2018 e PIB per capita 2018 (R$)
POP_REF = {"SP": 45538936, "MG": 21040662, "RJ": 17159960, "BA": 14812617, "PR": 11348937, "RS": 11329605,
           "PE": 9496294, "CE": 9075649, "PA": 8513497, "SC": 7075494, "MA": 7035055, "GO": 6921161,
           "AM": 4080611, "PB": 3996496, "ES": 3972388, "RN": 3479010, "MT": 3441998, "AL": 3322820,
           "PI": 3264531, "DF": 2974703, "MS": 2748023, "SE": 2278308, "RO": 1757589, "TO": 1555229,
           "AC": 869265, "AP": 829494, "RR": 576568}
PIBPC_REF = {"DF": 85661, "SP": 48542, "RJ": 44222, "SC": 42149, "RS": 40363, "MT": 39931, "MS": 38926,
             "PR": 38773, "ES": 34493, "MG": 29223, "GO": 28273, "RO": 26497, "AM": 24532, "RR": 23593,
             "TO": 22933, "AP": 20247, "BA": 19716, "PE": 19624, "RN": 19250, "PA": 18952, "SE": 18442,
             "AC": 17636, "CE": 17178, "AL": 16375, "PB": 16108, "PI": 15432, "MA": 13955}

def sidra_por_uf(tabela, variavel, ano=2018):
    url = f"https://apisidra.ibge.gov.br/values/t/{tabela}/n3/all/v/{variavel}/p/{ano}"
    with urllib.request.urlopen(url, timeout=20) as r:
        dados = json.load(r)
    cab, linhas = dados[0], pd.DataFrame(dados[1:])
    col_uf = next(k for k, v in cab.items() if str(v).startswith("Unidade da Federação (Código)"))
    s = pd.Series(pd.to_numeric(linhas["V"], errors="coerce").values, index=linhas[col_uf].astype(int).map(COD_UF))
    assert s.notna().sum() == 27
    return s

try:
    populacao = sidra_por_uf(6579, 9324)                 # população residente estimada
    pib = sidra_por_uf(5938, 37) * 1000                  # PIB a preços correntes (mil R$ → R$)
    pib_pc = pib / populacao
    assert pib_pc.between(5_000, 200_000).all(), "Valores do IBGE fora do esperado"
    fonte_ibge = "API SIDRA/IBGE (tabelas 6579 e 5938, 2018)"
except Exception as e:
    print("API do IBGE indisponível, usando valores de referência:", type(e).__name__)
    populacao, pib_pc = pd.Series(POP_REF), pd.Series(PIBPC_REF)
    fonte_ibge = "Valores de referência embutidos (aproximados), IBGE 2018"
registrar("externo", "Dados de população e PIB por UF", 27, f"Fonte: {fonte_ibge}",
          "Normalizar pedidos pelo tamanho e pela renda de cada estado")
print("Fonte:", fonte_ibge)

In [ ]:
uf_ext = uf.join(populacao.rename("populacao")).join(pib_pc.rename("pib_per_capita"))
uf_ext["pedidos_100k_hab"] = uf_ext["pedidos"] / uf_ext["populacao"] * 1e5

# Linha de tendência em escala log: quais estados compram menos do que a renda sugere?
x, y = np.log(uf_ext["pib_per_capita"]), np.log(uf_ext["pedidos_100k_hab"])
coef = np.polyfit(x, y, 1)
uf_ext["residuo"] = y - np.polyval(coef, x)
r = np.corrcoef(x, y)[0, 1]
abaixo = uf_ext.sort_values("residuo").head(3).index.tolist()
achados["ufs_subpenetradas"] = abaixo; achados["correlacao_renda_pedidos"] = float(r)

fig, ax = plt.subplots(figsize=(9, 5.5))
cores = [LARANJA if u in abaixo else AZUL for u in uf_ext.index]
ax.scatter(uf_ext["pib_per_capita"], uf_ext["pedidos_100k_hab"], s=70, color=cores, edgecolor="white", lw=1.5, zorder=3)
xs = np.linspace(x.min(), x.max(), 50)
ax.plot(np.exp(xs), np.exp(np.polyval(coef, xs)), color=CINZA, ls="--", lw=1.5, zorder=2)
for u, rw in uf_ext.iterrows():
    ax.annotate(u, (rw["pib_per_capita"], rw["pedidos_100k_hab"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xticks([15000, 20000, 30000, 40000, 60000, 80000])
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"R$ {num(v / 1000, 0)} mil"))
ax.xaxis.set_minor_formatter(mtick.NullFormatter())
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: num(v, 1) if v < 1 else num(v, 0)))
ax.set_title(f"Compras crescem com a renda; {', '.join(abaixo)} estão abaixo do esperado")
ax.set_xlabel("PIB per capita (2018, escala log)"); ax.set_ylabel("Pedidos por 100 mil hab. (escala log)")
salvar(fig, "04_penetracao_vs_renda")

## 9. Logística

O centro da tese: quanto tempo a entrega leva, em que etapa o tempo se perde e onde os atrasos se concentram.

In [ ]:
pct_atraso = base["atrasado"].mean()
achados.update(
    pct_atraso=float(pct_atraso), dias_entrega_mediana=float(base["dias_entrega"].median()),
    dias_entrega_media=float(base["dias_entrega"].mean()), dias_prometidos_media=float(base["dias_prometidos"].mean()),
    folga_media_no_prazo=float(-base.loc[~base["atrasado"], "dias_atraso"].mean()),
)
print(f"Pedidos atrasados: {pct(pct_atraso, 1)}")
print(f"Entrega: mediana de {num(achados['dias_entrega_mediana'])} dias | prazo prometido médio de {num(achados['dias_prometidos_media'])} dias")
print(f"Pedidos no prazo chegam, em média, {num(achados['folga_media_no_prazo'])} dias antes da data prevista")

In [ ]:
etapas = base.groupby("regiao_cliente")[["dias_aprovacao", "dias_postagem", "dias_transporte"]].mean()
etapas = etapas.loc[etapas.sum(axis=1).sort_values().index]
share_transp = base["dias_transporte"].mean() / base[["dias_aprovacao", "dias_postagem", "dias_transporte"]].mean().sum()
achados["share_tempo_transporte"] = float(share_transp)

fig, ax = plt.subplots(figsize=(9, 4))
esq = np.zeros(len(etapas))
for col, cor, nome in [("dias_aprovacao", CINZA, "Aprovação do pagamento"), ("dias_postagem", VERDE_AGUA, "Seller até a transportadora"),
                       ("dias_transporte", AZUL, "Transporte até o cliente")]:
    ax.barh(etapas.index, etapas[col], left=esq, color=cor, label=nome, edgecolor="white", lw=2)
    esq += etapas[col].values
for yy, tot in enumerate(esq):
    ax.text(tot + 0.2, yy, f"{num(tot)} dias", va="center", fontsize=9)
ax.legend(loc="lower right", frameon=False, fontsize=9); ax.grid(axis="y", visible=False)
ax.set_title(f"O transporte responde por {pct(share_transp)} do tempo de entrega")
ax.set_xlabel("Dias médios por etapa (região do cliente)")
salvar(fig, "05_tempo_por_etapa")

In [ ]:
atraso_uf = base.groupby("customer_state").agg(pct_atraso=("atrasado", "mean"), pedidos=("order_id", "count"))
atraso_uf = atraso_uf[atraso_uf["pedidos"] >= 30].sort_values("pct_atraso")  # evita taxas instáveis em UFs com poucos pedidos
pior = atraso_uf.index[-1]
achados.update(uf_mais_atraso=pior, pct_atraso_uf_pior=float(atraso_uf["pct_atraso"].iloc[-1]))

fig, ax = plt.subplots(figsize=(9, 7))
cores = [LARANJA if v > pct_atraso else CINZA for v in atraso_uf["pct_atraso"]]
ax.barh(atraso_uf.index, atraso_uf["pct_atraso"], color=cores)
ax.axvline(pct_atraso, color="#52514e", ls="--", lw=1)
ax.text(pct_atraso, len(atraso_uf) - 0.3, f" média {pct(pct_atraso, 1)}", fontsize=9, va="bottom")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0)); ax.grid(axis="y", visible=False)
ax.set_title(f"Atrasos chegam a {pct(atraso_uf['pct_atraso'].iloc[-1])} em {pior}, contra {pct(pct_atraso)} na média")
ax.set_xlabel("% de pedidos entregues após a data prevista (em laranja: acima da média)")
salvar(fig, "06_atraso_por_uf")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(range(len(m)), m["pct_atraso"], color=LARANJA, lw=2, marker="o", ms=4)
mes_pior = m["pct_atraso"].idxmax()
ax.set_xticks(range(len(m)), [rotulo_mes(x) for x in m.index], rotation=45)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0)); ax.set_ylim(0)
ax.set_title(f"A taxa de atraso oscila muito: pico de {pct(m['pct_atraso'].max())} em {rotulo_mes(mes_pior)}")
ax.set_ylabel("% de pedidos atrasados")
salvar(fig, "07_atraso_mensal")
achados.update(mes_pico_atraso=rotulo_mes(mes_pior), pct_atraso_mes_pico=float(m["pct_atraso"].max()))

In [ ]:
frete = base.groupby("regiao_cliente").agg(frete_pct=("frete_pct", "median"), dias=("dias_entrega", "median")).sort_values("frete_pct")
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(frete.index, frete["frete_pct"], color=[LARANJA if r in ("Norte", "Nordeste") else AZUL for r in frete.index])
for yy, (v, dd) in enumerate(zip(frete["frete_pct"], frete["dias"])):
    ax.text(v + 0.005, yy, f"{pct(v)} · entrega em {num(dd, 0)} dias", va="center", fontsize=9)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0)); ax.grid(axis="y", visible=False)
regiao_frete = frete.index[-1]
ax.set_title(f"No {regiao_frete}, o frete equivale a {pct(frete['frete_pct'].iloc[-1])} do preço (mediana)")
ax.set_xlabel("Frete como % do valor dos produtos (mediana por pedido)")
salvar(fig, "08_frete_por_regiao")
achados.update(regiao_frete_mais_caro=regiao_frete, frete_pct_regiao_mais_cara=float(frete["frete_pct"].iloc[-1]),
               frete_pct_sudeste=float(frete.loc["Sudeste", "frete_pct"]) if "Sudeste" in frete.index else None)

In [ ]:
# Rotas críticas: onde há MAIS atrasos do que o esperado pela média geral (rotas com 100+ pedidos).
# Ordenar só por número de atrasos destacaria apenas as rotas de maior volume.
rotas = base.groupby("rota").agg(pedidos=("order_id", "count"), atrasados=("atrasado", "sum"),
                                 pct_atraso=("atrasado", "mean"), dias_entrega=("dias_entrega", "mean"),
                                 nota=("review_score", "mean")).reset_index()
rotas["atrasos_esperados"] = rotas["pedidos"] * pct_atraso
rotas["atrasos_excedentes"] = rotas["atrasados"] - rotas["atrasos_esperados"]
rotas["share_atrasos"] = rotas["atrasados"] / rotas["atrasados"].sum()
rotas = rotas.sort_values("atrasos_excedentes", ascending=False)
top_rotas = rotas[rotas["pedidos"] >= 100].head(10)
share_top_rotas = top_rotas["atrasados"].sum() / rotas["atrasados"].sum()
share_ped_top_rotas = top_rotas["pedidos"].sum() / rotas["pedidos"].sum()
achados.update(share_atrasos_top10_rotas=float(share_top_rotas), share_pedidos_top10_rotas=float(share_ped_top_rotas),
               top_rotas=top_rotas["rota"].head(5).tolist(),
               pct_atraso_top10_rotas=float(top_rotas["atrasados"].sum() / top_rotas["pedidos"].sum()))

fig, ax = plt.subplots(figsize=(9, 4.8))
t = top_rotas.iloc[::-1]
ax.barh(t["rota"], t["pct_atraso"], color=LARANJA)
ax.axvline(pct_atraso, color="#52514e", ls="--", lw=1)
ax.text(pct_atraso, len(t) - 0.4, f" média {pct(pct_atraso, 1)}", fontsize=9, va="bottom")
for yy, (p_, n_) in enumerate(zip(t["pct_atraso"], t["pedidos"])):
    ax.text(p_ + 0.003, yy, f"{pct(p_)} · {num(n_, 0)} pedidos", va="center", fontsize=9)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0)); ax.grid(axis="y", visible=False)
ax.set_xlim(0, t["pct_atraso"].max() * 1.35)
ax.set_title(f"10 rotas somam {pct(share_top_rotas)} dos atrasos com só {pct(share_ped_top_rotas)} dos pedidos")
ax.set_xlabel("% de pedidos atrasados na rota (UF do seller → UF do cliente); rotas com mais atrasos acima da média")
salvar(fig, "09_rotas_criticas")
top_rotas[["rota", "pedidos", "atrasados", "pct_atraso", "atrasos_excedentes", "dias_entrega", "nota"]]

## 10. Satisfação

O atraso afeta a nota? O que os clientes insatisfeitos dizem?

In [ ]:
nota_grupo = base.groupby("atrasado")["review_score"].mean()
nota_prazo, nota_atraso = nota_grupo[False], nota_grupo[True]
baixa_grupo = base.groupby("atrasado")["nota_baixa"].mean()
achados.update(nota_media=float(base["review_score"].mean()), nota_no_prazo=float(nota_prazo), nota_atrasado=float(nota_atraso),
               pct_nota_baixa_no_prazo=float(baixa_grupo[False]), pct_nota_baixa_atrasado=float(baixa_grupo[True]))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(["Entregue no prazo", "Entregue com atraso"], [nota_prazo, nota_atraso], color=[AZUL, LARANJA], width=0.55)
for xx, v in enumerate([nota_prazo, nota_atraso]):
    ax.text(xx, v + 0.08, num(v), ha="center", fontsize=12, fontweight="bold")
ax.set_ylim(0, 5.4); ax.grid(axis="x", visible=False)
ax.set_title(f"Pedidos atrasados recebem nota {num(nota_atraso)}, contra {num(nota_prazo)} no prazo")
ax.set_ylabel("Nota média (1 a 5)")
salvar(fig, "10_nota_prazo_vs_atraso")

In [ ]:
faixa = base.groupby("faixa_atraso", observed=True).agg(nota=("review_score", "mean"), nota_baixa=("nota_baixa", "mean"), pedidos=("order_id", "count"))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(faixa.index.astype(str), faixa["nota_baixa"], color=[AZUL] + [LARANJA] * (len(faixa) - 1), width=0.6)
for xx, (v, n_) in enumerate(zip(faixa["nota_baixa"], faixa["pedidos"])):
    ax.text(xx, v + 0.015, pct(v), ha="center", fontsize=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0)); ax.grid(axis="x", visible=False)
ax.set_title(f"Com 15+ dias de atraso, {pct(faixa['nota_baixa'].iloc[-1])} dos clientes dão nota 1 ou 2")
ax.set_ylabel("% de avaliações com nota 1 ou 2"); ax.set_xlabel("Atraso em relação à data prevista")
salvar(fig, "11_nota_baixa_por_faixa_atraso")
achados["pct_nota_baixa_15mais"] = float(faixa["nota_baixa"].iloc[-1])
faixa

In [ ]:
# Que fatores mais derrubam a nota? Diferença da nota média entre o grupo "pior" e o restante
frete_alto = base["frete_pct"] > base["frete_pct"].quantile(0.75)
fatores = {
    "Entrega atrasada": base["atrasado"],
    "Pedido com 2+ sellers": base["qtd_sellers"] > 1,
    "Pedido com 2+ itens": base["qtd_itens"] > 1,
    "Frete alto (25% maiores)": frete_alto,
    "Envio interestadual": base["interestadual"],
}
impacto = pd.Series({k: base.loc[g, "review_score"].mean() - base.loc[~g, "review_score"].mean()
                     for k, g in fatores.items() if 30 <= g.sum() <= len(g) - 30}).sort_values()
achados["fator_maior_impacto"] = impacto.index[0]

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(impacto.index[::-1], impacto.values[::-1], color=[LARANJA if k == impacto.index[0] else CINZA for k in impacto.index[::-1]])
for yy, v in enumerate(impacto.values[::-1]):
    ax.text(v - 0.03 if v < 0 else v + 0.03, yy, num(v, 2), va="center", ha="right" if v < 0 else "left", fontsize=9)
ax.axvline(0, color="#52514e", lw=1); ax.grid(axis="y", visible=False)
ax.set_xlim(min(impacto.min() * 1.3, -0.5), max(impacto.max() + 0.3, 0.3))
ax.set_title(f"{impacto.index[0]} é o fator que mais derruba a nota")
ax.set_xlabel("Diferença na nota média vs. pedidos sem o fator (pontos)")
salvar(fig, "12_fatores_nota")

In [ ]:
# Simulação: quanto a nota média subiria se os atrasos caíssem?
def nota_simulada(reducao):
    """Pedidos atrasados que deixam de atrasar passam a ter a nota média dos pedidos no prazo."""
    s = base["review_score"].notna()
    share = base.loc[s, "atrasado"].mean()
    return base.loc[s, "review_score"].mean() + reducao * share * (nota_prazo - nota_atraso)

for r_ in (0.3, 0.5, 1.0):
    print(f"Reduzir os atrasos em {pct(r_)}: nota média de {num(achados['nota_media'], 2)} para {num(nota_simulada(r_), 2)}")
achados.update(nota_simulada_30pct=float(nota_simulada(0.3)), nota_simulada_sem_atraso=float(nota_simulada(1.0)))

In [ ]:
# Text mining: o que dizem as avaliações negativas (nota 1 ou 2)
try:
    import nltk
    nltk.download("stopwords", quiet=True)
    STOP = set(nltk.corpus.stopwords.words("portuguese"))
except Exception:
    STOP = set("a o e de da do das dos em no na nos nas um uma para por com que se não nao é foi ao aos à às mais mas "
               "muito me meu minha eu ele ela isso este esta ja já até ate ou sem seu sua pelo pela".split())

def sem_acento(t):
    return "".join(c for c in unicodedata.normalize("NFKD", t) if not unicodedata.combining(c))

STOP = {sem_acento(w) for w in STOP} | {"produto", "pedido", "loja", "compra", "ainda", "pois", "so", "dia", "dias"}
STOP -= {"nao"}  # "não" é central nas reclamações ("não recebi")

def tokens(t):
    t = sem_acento(str(t).lower())
    return [w for w in re.findall(r"[a-z]+", t) if len(w) > 2 or w == "nao"]

neg = base.loc[base["nota_baixa"] & base["review_comment_message"].notna(), "review_comment_message"]
palavras, bigramas = Counter(), Counter()
for txt in neg:
    tk = [w for w in tokens(txt) if w not in STOP]
    palavras.update(tk)
    bigramas.update(" ".join(b) for b in zip(tk, tk[1:]))

padrao_entrega = r"receb|entreg|cheg|prazo|atras|demor|correio|transport"
share_entrega = neg.map(lambda t: bool(re.search(padrao_entrega, sem_acento(str(t).lower())))).mean()
achados.update(pct_reclamacoes_sobre_entrega=float(share_entrega), comentarios_negativos=int(len(neg)))

top_p = pd.DataFrame(palavras.most_common(15), columns=["termo", "freq"])
top_b = pd.DataFrame(bigramas.most_common(15), columns=["termo", "freq"])
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, df_, tit in [(axs[0], top_p, "Palavras"), (axs[1], top_b, "Pares de palavras")]:
    ax.barh(df_["termo"][::-1], df_["freq"][::-1], color=AZUL); ax.set_title(tit, fontsize=11); ax.grid(axis="y", visible=False)
fig.suptitle(f"{pct(share_entrega)} dos comentários negativos falam de entrega", x=0.01, ha="left", fontweight="bold", fontsize=13)
fig.tight_layout()
salvar(fig, "13_termos_reviews_negativas")

## 11. Clientes: recompra, coortes e RFM

Os clientes voltam? A primeira experiência de entrega muda isso?

In [ ]:
clientes = base.groupby("customer_unique_id").agg(pedidos=("order_id", "count"), primeira=("order_purchase_timestamp", "min"),
                                                  ultima=("order_purchase_timestamp", "max"), valor=("valor_produtos", "sum")).reset_index()
taxa_recompra = (clientes["pedidos"] > 1).mean()
achados.update(clientes_unicos=int(len(clientes)), taxa_recompra=float(taxa_recompra))
print(f"Clientes únicos: {len(clientes):,} | compraram mais de uma vez: {pct(taxa_recompra, 1)}")

# Primeira compra de cada cliente: atrasou? nota? → voltou a comprar?
primeiro = (base.sort_values("order_purchase_timestamp").drop_duplicates("customer_unique_id")
                [["customer_unique_id", "atrasado", "review_score"]]
                .merge(clientes[["customer_unique_id", "pedidos"]], on="customer_unique_id"))
primeiro["voltou"] = primeiro["pedidos"] > 1
volta = primeiro.groupby("atrasado")["voltou"].mean()
achados.update(recompra_1a_no_prazo=float(volta.get(False, np.nan)), recompra_1a_atrasada=float(volta.get(True, np.nan)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(["1ª compra no prazo", "1ª compra atrasada"], [volta.get(False, 0), volta.get(True, 0)], color=[AZUL, LARANJA], width=0.55)
for xx, v in enumerate([volta.get(False, 0), volta.get(True, 0)]):
    ax.text(xx, v * 1.02, pct(v, 1), ha="center", fontsize=12, fontweight="bold")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=1)); ax.grid(axis="x", visible=False)
ax.set_ylim(0, max(volta.max(), 0.001) * 1.2)
ax.set_title(f"Só {pct(taxa_recompra, 1)} dos clientes voltam; após uma 1ª compra atrasada, {pct(volta.get(True, 0), 1)}")
ax.set_ylabel("% de clientes que fizeram uma 2ª compra")
salvar(fig, "14_recompra_por_primeira_entrega")

In [ ]:
# Coortes: % de clientes de cada mês de 1ª compra que compram de novo N meses depois
base["coorte"] = base.groupby("customer_unique_id")["mes"].transform("min")
base["mes_relativo"] = ((base["mes"].dt.year - base["coorte"].dt.year) * 12 + (base["mes"].dt.month - base["coorte"].dt.month))
tam = base.groupby("coorte")["customer_unique_id"].nunique()
ativos = base.groupby(["coorte", "mes_relativo"])["customer_unique_id"].nunique().reset_index(name="clientes")
ativos["tamanho_coorte"] = ativos["coorte"].map(tam)
ativos["pct_retido"] = ativos["clientes"] / ativos["tamanho_coorte"]
mat = ativos[(ativos["mes_relativo"].between(1, 12))].pivot(index="coorte", columns="mes_relativo", values="pct_retido")
mat.index = [rotulo_mes(i) for i in mat.index]
ret_max = np.nanmax(mat.values)
achados["retencao_mensal_maxima"] = float(ret_max)
achados["retencao_mes1_media"] = float(mat[1].mean()) if 1 in mat.columns else None

fig, ax = plt.subplots(figsize=(11, 6))
rotulos = mat.apply(lambda col: col.map(lambda v: pct(v, 1) if pd.notna(v) else ""))
sns.heatmap(mat, cmap="Blues", annot=rotulos, fmt="", annot_kws={"size": 7}, cbar=False, ax=ax, linewidths=1, linecolor="white")
ax.set_title(f"Nenhuma coorte retém mais de {pct(ret_max, 1)} dos clientes em um mês após a 1ª compra")
ax.set_xlabel("Meses após a primeira compra"); ax.set_ylabel("Mês da primeira compra")
salvar(fig, "15_coortes_retencao")

In [ ]:
# RFM: recência, frequência e valor por cliente
data_ref = base["order_purchase_timestamp"].max() + pd.Timedelta(days=1)
rfm = clientes.assign(recencia=(data_ref - clientes["ultima"]).dt.days, frequencia=clientes["pedidos"], monetario=clientes["valor"])
rfm["R"] = pd.qcut(rfm["recencia"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)  # 5 = mais recente
rfm["M"] = pd.qcut(rfm["monetario"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segmento(r):
    if r["frequencia"] >= 2 and r["R"] >= 4: return "Campeões"
    if r["frequencia"] >= 2: return "Leais"
    if r["R"] >= 4: return "Novos (1 compra recente)"
    if r["M"] >= 4: return "Em risco (alto valor)"
    return "Hibernando"

rfm["segmento"] = rfm.apply(segmento, axis=1)
seg = rfm.groupby("segmento").agg(clientes=("customer_unique_id", "count"), receita=("monetario", "sum"))
seg["pct_clientes"] = seg["clientes"] / seg["clientes"].sum(); seg["pct_receita"] = seg["receita"] / seg["receita"].sum()
ordem = ["Campeões", "Leais", "Novos (1 compra recente)", "Em risco (alto valor)", "Hibernando"]
seg = seg.reindex([s for s in ordem if s in seg.index])
achados["pct_clientes_uma_compra"] = float((rfm["frequencia"] == 1).mean())
achados["segmentos_rfm"] = seg[["pct_clientes", "pct_receita"]].round(4).to_dict("index")

fig, ax = plt.subplots(figsize=(9, 4))
yy = np.arange(len(seg))
ax.barh(yy + 0.2, seg["pct_clientes"], 0.38, color=AZUL, label="% dos clientes")
ax.barh(yy - 0.2, seg["pct_receita"], 0.38, color=VERDE_AGUA, label="% da receita")
ax.set_yticks(yy, seg.index); ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0)); ax.legend(frameon=False, loc="lower right")
ax.set_title(f"{pct(achados['pct_clientes_uma_compra'])} dos clientes compraram uma única vez")
salvar(fig, "16_segmentos_rfm")
seg

## 12. Previsão de pedidos

Três modelos são comparados num **teste retroativo**: cada um é treinado sem os últimos 6 meses e comparado com o que de fato aconteceu. O de menor erro médio (MAPE) faz a previsão dos próximos 6 meses, com intervalo de 80%.

**Limitações:** a série tem apenas 20 meses, então a sazonalidade anual (como a Black Friday) não é captada; e agosto/2018 pode estar levemente subestimado, porque pedidos do fim do mês ainda estavam em trânsito quando os dados foram extraídos.

In [ ]:
try:
    from prophet import Prophet
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "prophet"], check=False)
    from prophet import Prophet
from statsmodels.tsa.statespace.exponential_smoothing import ExponentialSmoothing as SuavizacaoExp
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

serie = mensal[["mes", "pedidos"]].rename(columns={"mes": "ds", "pedidos": "y"}).reset_index(drop=True)
HORIZONTE = 6

def prever_prophet(df, h):
    mdl = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False, interval_width=0.8)
    mdl.fit(df)
    p_ = mdl.predict(mdl.make_future_dataframe(periods=h, freq="MS")).tail(h)
    return p_["yhat"].values, p_["yhat_lower"].values, p_["yhat_upper"].values

def prever_suavizacao(tendencia):
    def f(df, h):
        mdl = SuavizacaoExp(df["y"].astype(float).values, trend=tendencia, damped_trend=tendencia).fit(disp=False)
        fc = mdl.get_forecast(h)
        ic = np.asarray(fc.conf_int(alpha=0.2))
        return np.asarray(fc.predicted_mean), ic[:, 0], ic[:, 1]
    return f

MODELOS = {
    "Prophet (tendência linear)": prever_prophet,
    "Holt com tendência amortecida": prever_suavizacao(True),
    "Suavização exponencial simples": prever_suavizacao(False),
}

# Teste retroativo (backtest)
treino, teste = serie.iloc[:-HORIZONTE], serie.iloc[-HORIZONTE:]
erros = {}
for nome, f in MODELOS.items():
    yhat_teste, _, _ = f(treino, HORIZONTE)
    erros[nome] = float(np.mean(np.abs(yhat_teste - teste["y"].values) / teste["y"].values))
erros = pd.Series(erros).sort_values()
print("Erro médio nos 6 meses de teste (MAPE):")
print(erros.map(lambda e: pct(e, 1)).to_string())

# Previsão final com o melhor modelo e toda a série
modelo_escolhido = erros.index[0]
mape = erros.iloc[0]
yhat, lo, hi = MODELOS[modelo_escolhido](serie, HORIZONTE)
datas_fut = pd.date_range(serie["ds"].max() + pd.offsets.MonthBegin(1), periods=HORIZONTE, freq="MS")
prev_fut = pd.DataFrame({"ds": datas_fut, "yhat": yhat, "yhat_lower": lo, "yhat_upper": hi})
prev = pd.concat([serie.rename(columns={"y": "yhat"}).tail(1), prev_fut], ignore_index=True)  # liga o último mês real à previsão
print(f"Modelo escolhido: {modelo_escolhido}")
achados.update(modelo_previsao=modelo_escolhido, mape_backtest=float(mape), erros_modelos=erros.round(4).to_dict(),
               previsao_6m_total=float(prev_fut["yhat"].sum()), previsao_6m_media_mensal=float(prev_fut["yhat"].mean()),
               media_mensal_ultimos_6m=float(serie["y"].tail(6).mean()),
               media_mensal_2018=float(serie.loc[serie["ds"] >= "2018-01-01", "y"].mean()))

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(serie["ds"], serie["y"], color=AZUL, lw=2, marker="o", ms=4, label="Realizado")
ax.plot(prev["ds"], prev["yhat"], color=LARANJA, lw=2, ls="--", marker="o", ms=4, label=f"Previsto ({modelo_escolhido.lower()})")
ax.fill_between(prev_fut["ds"], prev_fut["yhat_lower"], prev_fut["yhat_upper"], color=LARANJA, alpha=0.15, label="Intervalo de 80%")
todas = list(serie["ds"]) + list(datas_fut)
ax.set_xticks(todas, [rotulo_mes(x) for x in todas], rotation=45); ax.set_ylim(0)
ax.legend(frameon=False, loc="upper left")
ax.set_title(f"Previsão: cerca de {num(prev_fut['yhat'].mean(), 0)} pedidos/mês até {rotulo_mes(datas_fut[-1])} (erro no teste: {pct(mape)})")
ax.set_ylabel("Pedidos entregues por mês")
salvar(fig, "17_previsao_pedidos")
prev_fut

## 13. Exportação para o Power BI e documentação

Os CSVs usam `;` como separador e `,` como decimal (padrão brasileiro do Power BI) e codificação UTF-8.

In [ ]:
def exportar(df, nome):
    df = df.copy()
    decimais = df.select_dtypes("float").columns
    df[decimais] = df[decimais].round(4)  # evita "lixo" de ponto flutuante (ex.: 75,99000000000001) que confunde o Power BI
    df.to_csv(os.path.join(PASTA_PROCESSED, f"{nome}.csv"), sep=";", decimal=",", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
    print(f"{nome}.csv: {len(df):,} linhas")

fato = base.rename(columns={
    "order_id": "pedido_id", "customer_unique_id": "cliente_id", "customer_state": "uf_cliente", "customer_city": "cidade_cliente",
    "seller_id": "seller_id", "seller_state": "uf_seller", "order_purchase_timestamp": "data_compra",
    "order_delivered_customer_date": "data_entrega", "order_estimated_delivery_date": "data_prevista",
    "review_score": "nota"})
fato["data_compra"] = fato["data_compra"].dt.date; fato["data_entrega"] = fato["data_entrega"].dt.date
fato["data_prevista"] = fato["data_prevista"].dt.date
fato["status_entrega"] = np.where(fato["atrasado"], "Atrasado", "No prazo")
COLS_FATO = ["pedido_id", "cliente_id", "data_compra", "mes", "data_entrega", "data_prevista", "uf_cliente", "estado_cliente",
             "regiao_cliente", "uf_seller", "regiao_seller", "rota", "interestadual", "seller_id", "categoria", "qtd_itens",
             "qtd_sellers", "valor_produtos", "valor_frete", "frete_pct", "valor_pago", "forma_pagamento", "parcelas",
             "dias_entrega", "dias_aprovacao", "dias_postagem", "dias_transporte", "dias_prometidos", "dias_atraso",
             "atrasado", "status_entrega", "faixa_atraso", "nota", "nota_baixa", "coorte"]
fato["estado_cliente"] = fato["uf_cliente"].map(NOME_UF) + ", Brasil"  # ajuda o mapa do Power BI
exportar(fato[COLS_FATO], "fato_pedidos")

uf_ind = (base.groupby("customer_state").agg(pedidos=("order_id", "count"), receita=("valor_produtos", "sum"),
                                             ticket_medio=("valor_produtos", "mean"), pct_atraso=("atrasado", "mean"),
                                             nota_media=("review_score", "mean"), dias_entrega_medio=("dias_entrega", "mean"),
                                             frete_pct_mediano=("frete_pct", "median"))
                .join(uf_ext[["populacao", "pib_per_capita", "pedidos_100k_hab"]]).reset_index()
                .rename(columns={"customer_state": "uf"}))
uf_ind.insert(1, "estado", uf_ind["uf"].map(NOME_UF) + ", Brasil"); uf_ind.insert(2, "regiao", uf_ind["uf"].map(REGIAO))
exportar(uf_ind, "uf_indicadores")

exportar(rfm[["customer_unique_id", "recencia", "frequencia", "monetario", "R", "M", "segmento"]].rename(columns={"customer_unique_id": "cliente_id"}), "rfm_clientes")
exportar(ativos.rename(columns={"coorte": "mes_primeira_compra"}), "coortes")
exportar(pd.concat([top_p.assign(tipo="Palavra"), top_b.assign(tipo="Par de palavras")]), "termos_reviews")
exportar(rotas, "rotas")
exportar(serie.merge(prev, on="ds", how="outer").rename(columns={"ds": "mes", "y": "realizado", "yhat": "previsto",
                                                                  "yhat_lower": "previsto_min", "yhat_upper": "previsto_max"}), "previsao")
exportar(registro_df, "registro_qualidade")

In [ ]:
# Documentação: registro de qualidade e dicionário de dados em Markdown (para o repositório)
def tabela_md(df):
    linhas = ["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)]
    linhas += ["| " + " | ".join(str(v) for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join(linhas)

with open(os.path.join(PASTA_DOCS, "qualidade_dados.md"), "w", encoding="utf-8") as f:
    f.write("# Registro de qualidade e decisões\n\n")
    f.write(f"Período analisado: {DATA_INICIO} a {DATA_FIM} (data da compra). Base final: {num(len(base), 0)} pedidos entregues.\n\n")
    f.write(f"Definição de receita: soma do preço dos itens, sem frete. Dados externos: {fonte_ibge}.\n\n")
    f.write(tabela_md(registro_df))

DICIONARIO = {
    "pedido_id": "Identificador do pedido", "cliente_id": "Identificador único da pessoa (customer_unique_id)",
    "data_compra": "Data da compra", "mes": "Mês da compra", "data_entrega": "Data de entrega ao cliente",
    "data_prevista": "Data de entrega prometida", "uf_cliente": "UF do cliente", "estado_cliente": "Nome do estado (para mapas)",
    "regiao_cliente": "Região do cliente", "uf_seller": "UF do seller principal", "regiao_seller": "Região do seller",
    "rota": "UF do seller → UF do cliente", "interestadual": "Seller e cliente em UFs diferentes",
    "seller_id": "Seller do item mais caro do pedido", "categoria": "Categoria do item mais caro do pedido",
    "qtd_itens": "Número de itens", "qtd_sellers": "Número de sellers distintos", "valor_produtos": "Receita: soma do preço dos itens (R$)",
    "valor_frete": "Soma do frete (R$)", "frete_pct": "Frete ÷ valor dos produtos", "valor_pago": "Soma dos pagamentos (R$)",
    "forma_pagamento": "Forma de pagamento de maior valor", "parcelas": "Maior número de parcelas",
    "dias_entrega": "Dias da compra à entrega", "dias_aprovacao": "Dias da compra à aprovação do pagamento",
    "dias_postagem": "Dias da aprovação à entrega à transportadora", "dias_transporte": "Dias da transportadora ao cliente",
    "dias_prometidos": "Dias da compra à data prometida", "dias_atraso": "Dias entre a entrega e a data prometida (positivo = atraso)",
    "atrasado": "Entregue após a data prometida", "status_entrega": "No prazo / Atrasado", "faixa_atraso": "Faixa de dias de atraso",
    "nota": "Nota da avaliação (1 a 5)", "nota_baixa": "Nota 1 ou 2", "coorte": "Mês da primeira compra do cliente",
}
with open(os.path.join(PASTA_DOCS, "dicionario_dados.md"), "w", encoding="utf-8") as f:
    f.write("# Dicionário de dados: fato_pedidos.csv\n\nUma linha por pedido entregue no período analisado.\n\n")
    f.write(tabela_md(pd.DataFrame({"coluna": list(DICIONARIO), "descricao": list(DICIONARIO.values())})))

with open("requirements.txt", "w") as f:
    f.write("pandas\nnumpy\nmatplotlib\nseaborn\nkagglehub\nprophet\nstatsmodels\nnltk\n")
print("Documentação salva em docs/ e requirements.txt")

## 14. Principais números para o relatório

Todos os números citados nos slides e no vídeo saem daqui (também salvos em `reports/achados.json`).

In [ ]:
with open("reports/achados.json", "w", encoding="utf-8") as f:
    json.dump(achados, f, ensure_ascii=False, indent=2, default=str)

a = achados
print(f"""
CRESCIMENTO
  {num(a['pedidos_total'], 0)} pedidos entregues e R$ {num(a['receita_total'] / 1e6, 1)} mi em receita (jan/17–ago/18)
  Pedidos +{pct(a['crescimento_pedidos_yoy'])} e receita +{pct(a['crescimento_receita_yoy'])} (jan–ago/18 vs jan–ago/17)
  {a['top_uf']} = {pct(a['share_receita_top_uf'])} da receita | Sudeste = {pct(a['share_receita_sudeste'])} | top 10 categorias = {pct(a['share_receita_top10_categorias'])}
  Estados abaixo do potencial de renda: {', '.join(a['ufs_subpenetradas'])}

LOGÍSTICA
  {pct(a['pct_atraso'], 1)} dos pedidos atrasam | pior UF: {a['uf_mais_atraso']} ({pct(a['pct_atraso_uf_pior'])})
  Entrega leva {num(a['dias_entrega_mediana'])} dias (mediana); prazo prometido médio de {num(a['dias_prometidos_media'])} dias
  Transporte = {pct(a['share_tempo_transporte'])} do tempo de entrega
  10 rotas críticas (atraso acima da média) = {pct(a['share_atrasos_top10_rotas'])} dos atrasos e só {pct(a['share_pedidos_top10_rotas'])} dos pedidos
  Frete no {a['regiao_frete_mais_caro']}: {pct(a['frete_pct_regiao_mais_cara'])} do preço

SATISFAÇÃO
  Nota {num(a['nota_no_prazo'])} no prazo vs {num(a['nota_atrasado'])} com atraso
  Notas 1–2: {pct(a['pct_nota_baixa_no_prazo'])} no prazo vs {pct(a['pct_nota_baixa_atrasado'])} com atraso ({pct(a['pct_nota_baixa_15mais'])} com 15+ dias)
  {pct(a['pct_reclamacoes_sobre_entrega'])} das reclamações falam de entrega
  Reduzir atrasos em 30%: nota média de {num(a['nota_media'], 2)} para {num(a['nota_simulada_30pct'], 2)}

CLIENTES
  Só {pct(a['taxa_recompra'], 1)} voltam a comprar | recompra após 1ª compra no prazo {pct(a['recompra_1a_no_prazo'], 1)} vs atrasada {pct(a['recompra_1a_atrasada'], 1)}
  {pct(a['pct_clientes_uma_compra'])} dos clientes compraram uma única vez

PREVISÃO
  ~{num(a['previsao_6m_media_mensal'], 0)} pedidos/mês nos próximos 6 meses (média de 2018: {num(a['media_mensal_2018'], 0)})
  Modelo: {a['modelo_previsao']} | erro no teste: {pct(a['mape_backtest'], 1)}
""")

In [ ]:
# Empacota tudo para baixar (no Colab, o download começa sozinho)
pacote = "olist_resultados"
os.makedirs(pacote, exist_ok=True)
for pasta in ("data", "reports", "docs"):
    shutil.copytree(pasta, os.path.join(pacote, pasta), dirs_exist_ok=True)
shutil.copy("requirements.txt", pacote)
shutil.make_archive(pacote, "zip", pacote)
try:
    from google.colab import files
    files.download(f"{pacote}.zip")
except ImportError:
    print(f"Arquivo gerado: {pacote}.zip")